# Spatial Panel Time-Series Analysis of Urban Morphology  
## Subdivision–NPA Aggregation and Fixed-Effects Modeling (1990–2023)

This notebook constructs a **spatio-temporal panel dataset** by aggregating
subdivision-level urban morphology indicators to **dominant Neighborhood Profile Areas (NPAs)**.

The workflow includes:
- Spatial intersection and area-weighted attribution of subdivisions to NPAs
- Construction of dominance and fragmentation indicators
- Panel aggregation and diagnostics
- Fixed-effects, two-way fixed-effects, and dynamic panel models
- Structural break detection in long-run trends

The final objective is to assess **temporal trends and regime shifts** in urban morphology
while controlling for unobserved spatial heterogeneity.


In [1]:
import pandas as pd
import geopandas as gpd
import folium
from arch.unitroot import ADF, KPSS
from linearmodels.panel import PanelOLS
import statsmodels.api as sm
import ruptures as rpt
import numpy as np
from IPython.display import HTML

pd.set_option("display.float_format", "{:.4f}".format)

In [2]:
ABT = gpd.read_file(
    "../../../../Data/Final_dataset/ABT/ABT.gpkg",
    layer="subdivisions"
)

npa_raw = gpd.read_file(
    "../../../../Data/Original_dataset/original.gdb",
    layer="QOL_NPA_2020_final_projected"
)

In [3]:
ABT_proj = ABT[ABT["year"].between(1990, 2023)].copy()
npa_proj = npa_raw.to_crs(ABT_proj.crs)

ABT_proj["subd_area"] = ABT_proj.geometry.area

## 1. Spatial Overlay: Subdivision × NPA

Each subdivision may intersect multiple NPAs.
We compute intersection geometries and their respective areas.


In [4]:
abt_npa_intersections = gpd.overlay(
    ABT_proj[["subd_id", "geometry"]],
    npa_proj[["NPA_ID", "geometry"]],
    how="intersection"
)

abt_npa_intersections["intersect_area"] = (
    abt_npa_intersections.geometry.area
)

## 2. Area Shares, NPA Counts, and Fragmentation Structure

For each subdivision:
- Count how many NPAs it intersects
- Compute area-weighted shares
- Rank NPAs by dominance


In [5]:
npa_area = (
    abt_npa_intersections
    .groupby(["subd_id", "NPA_ID"])["intersect_area"]
    .sum()
    .reset_index()
)

npa_count = (
    npa_area
    .groupby("subd_id")["NPA_ID"]
    .nunique()
    .reset_index(name="npa_count")
)

npa_id_list = (
    npa_area
    .groupby("subd_id")["NPA_ID"]
    .apply(lambda x: sorted(x.unique().tolist()))
    .reset_index(name="npa_id_list")
)

npa_id_list

,subd_id,npa_id_list
0,2,[273.0]
1,3,"[301.0, 377.0]"
2,4,[299.0]
3,8,[299.0]
4,15,[273.0]
...,...,...
5839,10179,[396.0]
5840,10180,[396.0]
5841,10181,[396.0]
5842,10182,[396.0]


## 3. Dominant NPA Ranking and Share Validation

We compute the proportional area share of each NPA per subdivision
and verify that all shares sum to unity.


In [6]:
npa_area = npa_area.merge(
    ABT_proj[["subd_id", "subd_area"]],
    on="subd_id",
    how="left"
)

npa_area["share"] = (
    npa_area["intersect_area"] / npa_area["subd_area"]
)

npa_area = npa_area.sort_values(
    ["subd_id", "share"],
    ascending=[True, False]
)

npa_area["rank"] = (
    npa_area
    .groupby("subd_id")
    .cumcount() + 1
)

## 4. Wide-Format Share Matrix (Top 6 NPAs)

To maintain interpretability and avoid extreme fragmentation,
we retain the top 6 NPAs per subdivision and aggregate the remainder.


In [7]:
npa_area_top = npa_area[npa_area["rank"] <= 6].copy()

npa_share_wide = (
    npa_area_top
    .pivot_table(
        index="subd_id",
        columns="rank",
        values="share",
        fill_value=0
    )
)

npa_share_wide.columns = [
    f"share_npa_{int(c)}" for c in npa_share_wide.columns
]

npa_share_wide = npa_share_wide.reset_index()

npa_share_wide

,subd_id,share_npa_1,share_npa_2,share_npa_3,share_npa_4,share_npa_5,share_npa_6
0,2,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,3,0.9927,0.0073,0.0000,0.0000,0.0000,0.0000
2,4,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000
3,8,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,15,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000
...,...,...,...,...,...,...,...
5839,10179,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000
5840,10180,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000
5841,10181,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000
5842,10182,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000


## 5. Final Share Assembly and Consistency Check

We compute the residual share outside NPAs and verify numerical integrity.


In [8]:
ABT_proj = (
    ABT_proj
    .merge(npa_share_wide, on="subd_id", how="left")
    .merge(npa_count, on="subd_id", how="left")
    .merge(npa_id_list, on="subd_id", how="left")
)

share_cols = [f"share_npa_{k}" for k in range(1, 7)]

ABT_proj[share_cols] = ABT_proj[share_cols].fillna(0)
ABT_proj["npa_count"] = ABT_proj["npa_count"].fillna(0).astype(int)

ABT_proj["share_outside_npa"] = (
    1 - ABT_proj[share_cols].sum(axis=1)
).clip(lower=0)

ABT_proj["share_sum_check"] = (
    ABT_proj[share_cols + ["share_outside_npa"]].sum(axis=1)
)

assert ABT_proj["share_sum_check"].between(0.999, 1.001).all()

## 6. NPA Attribution Strategy: Area-Weighted Contribution

Instead of assigning each subdivision to a single “dominant” NPA, we follow an
**area-weighted contribution rule**:

- A subdivision can contribute to multiple NPAs if it overlaps multiple polygons.
- We do **not** split the indicator value itself (e.g., FAR, accessibility scores).
- We compute each NPA-year value as a **weighted mean** of subdivision scores,
  where weights are proportional to the subdivision’s area inside that NPA.


In [9]:
# Build a (subd_id, NPA_ID) weight table from the overlay results
# Option A (recommended): use intersection area as weights
weights = npa_area[["subd_id", "NPA_ID", "intersect_area", "share"]].copy()

# Choose your weight definition:
#   - intersect_area: absolute area inside the NPA (preferred for area-weighted mean)
#   - share: fraction of subdivision area inside the NPA (also valid)
WEIGHT_COL = "intersect_area"   # or "share"
weights = weights.rename(columns={"NPA_ID": "npa_id"})

## 7. Panel Construction (NPA × Year) Using Area-Weighted Means

We construct the NPA-year panel by:

1. Merging subdivision-year indicators with the subdivision-NPA weight table  
2. Computing an area-weighted mean for each indicator within each (NPA, year)

Formally, for indicator \(x\):

$$
\bar{x}_{jt} = \frac{\sum_{s} w_{sjt}\, x_{st}}{\sum_{s} w_{sjt}}
$$

where \(w_{sjt}\) denotes the overlap weight between subdivision \(s\) and NPA \(j\) in year \(t\).

**Thus, each NPA-year value is an area-weighted mean of subdivision-level values, where weights reflect the proportion (or area) of each subdivision contained within the NPA.**


In [10]:
# 1) Select variables to aggregate (subdivision-year level)
vars_to_agg = [
    'HAC_dist', 'BAD', 'SHD',
    'int_den025','nd_deg025', 'int_den05', 'nd_deg05',
    'int_den075','nd_deg075', 'int_den1', 'nd_deg1',
    'AI', 'PROX', 'ENN_MN', 'ED', 'SHAPE_MN', 'FRAC_MN',
    'ENN_inv', 'ED_inv', 'SHAPE_inv', 'FRAC_inv',
    'AI_norm', 'PROX_norm', 'ENN_inv_norm', 'ED_inv_norm',
    'SHAPE_inv_norm', 'FRAC_inv_norm',
    'COMPACTNESS_SUM', 'BAD_ctx_025', 'BAD_ctx_050',
    'groceries_ws', 'transit_ws', 'FAR'
]

sub_year = ABT_proj[["subd_id", "year"] + vars_to_agg].copy()

# 2) Attach weights (subd_id x npa_id)
df = sub_year.merge(weights[["subd_id", "npa_id", WEIGHT_COL]], on="subd_id", how="left")

# Safety: drop cases with missing weights (should be rare; indicates outside-NPA-only subs)
df = df.dropna(subset=[WEIGHT_COL])

# 3) Weighted mean helper
def weighted_mean(g, value_col, w_col):
    w = g[w_col].to_numpy()
    x = g[value_col].to_numpy()
    denom = w.sum()
    return (w * x).sum() / denom if denom != 0 else np.nan

# 4) Compute NPA-year weighted means for all indicators
panel_weighted = (
    df.groupby(["npa_id", "year"])
      .apply(lambda g: pd.Series({v: weighted_mean(g, v, WEIGHT_COL) for v in vars_to_agg}))
      .sort_index()
)

panel = panel_weighted.copy()
panel.index = panel.index.set_names(["NPA_ID", "year"])

C:\Users\erfan\AppData\Local\Temp\ipykernel_3276\3982791484.py:32: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({v: weighted_mean(g, v, WEIGHT_COL) for v in vars_to_agg}))


In [11]:
panel

HAC_dist    BAD    SHD  int_den025  nd_deg025  int_den05  \
NPA_ID   year                                                                  
2.0000   1996.0000    1.8000 0.1940 0.4000      0.1164     2.0465     0.0845   
         1999.0000    1.5060 0.2105 0.0000      0.1007     2.0091     0.0868   
         2001.0000    1.4000 0.1410 0.0000      0.1124     2.0465     0.0920   
         2008.0000    1.8600 0.2120 0.2200      0.1011     2.0690     0.0823   
         2009.0000    2.0200 0.2290 0.0000      0.0818     2.0741     0.0882   
...                      ...    ...    ...         ...        ...        ...   
476.0000 2018.0000    0.4950 0.4969 0.8650      0.1622     2.4615     0.1496   
         2019.0000    0.3773 0.5476 0.6875      0.1656     2.5822     0.1697   
         2020.0000    0.3200 0.7290 1.0000      0.2111     2.6333     0.1799   
         2022.0000    0.5353 0.4661 0.4592      0.2218     2.5617     0.1886   
         2023.0000    0.6120 0.2142 0.2472      0.0798     2.3202     0.1003   

                    nd_deg05  int_den075  nd_deg075  int_den1  ...  \
NPA_ID   year                                                  ...   
2.0000   1996.0000    2.1702      0.1013     2.3568    0.1020  ...   
         1999.0000    2.2022      0.1001     2.3224    0.1226  ...   
         2001.0000    2.1942      0.1019     2.3223    0.1374  ...   
         2008.0000    2.1628      0.1036     2.3689    0.1049  ...   
         2009.0000    2.3133      0.1158     2.3556    0.1043  ...   
...                      ...         ...        ...       ...  ...   
476.0000 2018.0000    2.6813      0.1368     2.7444    0.1297  ...   
         2019.0000    2.7695      0.1752     2.8097    0.1632  ...   
         2020.0000    2.8333      0.1414     2.8046    0.1358  ...   
         2022.0000    2.6860      0.1688     2.8206    0.1679  ...   
         2023.0000    2.5821      0.1544     2.6050    0.1496  ...   

                    ENN_inv_norm  ED_inv_norm  SHAPE_inv_norm  FRAC_inv_norm  \
NPA_ID   year                                                                  
2.0000   1996.0000        0.1054       0.0026          0.4997         0.6063   
         1999.0000        0.0619       0.0033          0.5475         0.6408   
         2001.0000        0.0521       0.0053          0.6125         0.6834   
         2008.0000        0.0483       0.0022          0.8772         0.8751   
         2009.0000        0.1304       0.0019          0.8757         0.8644   
...                          ...          ...             ...            ...   
476.0000 2018.0000        0.7821       0.0117          0.6907         0.7873   
         2019.0000        0.2093       0.0061          0.8808         0.8826   
         2020.0000        1.0000       0.0031          0.5342         0.7091   
         2022.0000        0.4887       0.0040          0.7936         0.8383   
         2023.0000        0.0195       0.0073          0.8914         0.8969   

                    COMPACTNESS_SUM  BAD_ctx_025  BAD_ctx_050  groceries_ws  \
NPA_ID   year                                                                 
2.0000   1996.0000           0.0479       0.1280       0.1250       48.7400   
         1999.0000           0.0369       0.1193       0.1246       56.7701   
         2001.0000           0.0342       0.1100       0.1310       60.1000   
         2008.0000           0.0301       0.1110       0.1190       29.9200   
         2009.0000           0.0582       0.0970       0.1060       41.6900   
...                             ...          ...          ...           ...   
476.0000 2018.0000           0.3267       0.2491       0.2437       86.8555   
         2019.0000           0.1274       0.3505       0.2976       74.5353   
         2020.0000           0.4309       0.2350       0.2720       96.4300   
         2022.0000           0.2076       0.2445       0.2619       79.7082   
         2023.0000           0.0341       0.1188       0.1540       58.5431   


In [12]:
# !jupyter nbconvert --to html --no-input EDA2.ipynb --output ../../../../output/Notebook_Outputs/spatio_temporal/EDA2.html

[NbConvertApp] Converting notebook EDA2.ipynb to html
[NbConvertApp] Writing 595229 bytes to ..\..\..\..\output\Notebook_Outputs\spatio_temporal\EDA2.html
